# Geotag: full pipeline on Colab (Qwen2.5-VL-3B, L4 GPU)

Runs every GPU step from scratch and scores all TARA test images. Before the first run:

1. **Runtime → Change runtime type → L4 GPU.** (A100 now needs a Google AI Ultra plan; the 3B model fits in L4's 24 GB.)
2. Google Drive has `My Drive/geotag/` with `data/tara_img.zip`, `runs_3b/prepared/` and `runs_3b/geonames/`.
3. The GitHub repo is private, so add a fine-grained, **read-only** token for `geotag` in Colab **Secrets** (key icon on the left) under the name `GH_TOKEN`, with notebook access on.

Every step writes to `My Drive/geotag/runs_3b/` and is skipped when its output is already there. After a disconnect, run all cells again and it continues where it stopped (a half-finished LoRA training restarts from step 0).

Rough cost (estimate): about 4–5 L4 hours in total, about 1.7 compute units per hour.

In [14]:
gpu = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits
print(gpu[0])
mem = gpu[0].split(",")[-1].strip()
if not mem.isdigit() or int(mem) < 22000:
    print("WARNING: needs a GPU with at least 22 GB. Runtime -> Change runtime type -> L4 GPU")

Tesla T4, 15360


In [15]:
import os
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/geotag"
CODE = "/content/geotag"
REPO = "https://github.com/Tanny-24/geotag.git"
for f in ["data/tara_img.zip", "runs_3b/prepared/corpus.jsonl", "runs_3b/geonames/cities15000.txt"]:
    assert os.path.exists(f"{DRIVE}/{f}"), f"missing on Drive: {f}"
print("Drive OK")

Mounted at /content/drive
Drive OK


## 0. Code from GitHub, packages, images

The token is passed to git through environment variables for this one command, so it never lands in a URL, in `.git/config` or in an error message.

In [16]:
import base64
import subprocess
from google.colab import userdata

try:
    token = userdata.get("GH_TOKEN")
except Exception:              # no secret: only works once the repo is public
    token = None
env = dict(os.environ)
if token:
    basic = base64.b64encode(f"Tanny-24:{token}".encode()).decode()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
               GIT_CONFIG_VALUE_0=f"Authorization: Basic {basic}")
git = ["git", "-C", CODE, "pull", "-q"] if os.path.isdir(CODE) else ["git", "clone", "-q", REPO, CODE]
subprocess.run(git, env=env, check=True)
COMMIT = subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print("code at", COMMIT)

code at a1b2ec1 Add evidence-aware confidence and a re-split check to the trust layer


In [17]:
!pip install -q -r {CODE}/requirements.txt
!mkdir -p /content/data/TARA_dataset && unzip -q -n {DRIVE}/data/tara_img.zip -d /content/data/TARA_dataset
for s in ("train", "dev", "test"):
    print(s, len(os.listdir(f"/content/data/TARA_dataset/img/{s}")))   # expect 11241 / 1644 / 1644

train 11241
dev 1644
test 1644


In [18]:
import json

os.environ.update(NV_DATA="/content/data", NV_OUT=f"{DRIVE}/runs_3b", CUDA_VISIBLE_DEVICES="0",
                  PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
OUT = os.environ["NV_OUT"]
os.makedirs(f"{OUT}/logs", exist_ok=True)
os.chdir(CODE)


def step(cmd, done=None, n=None, log=None):
    """Run one pipeline command and save its output to runs_3b/logs/. Skip it when `done` (a path under
    runs_3b/) exists and, if `n` is given, has n lines: infer writes its file batch by batch, so a
    half-finished run is redone instead of skipped."""
    path = f"{OUT}/{done}" if done else None
    if path and os.path.exists(path) and (n is None or sum(1 for _ in open(path)) == n):
        print("skip, already done:", cmd)
        return
    with open(f"{OUT}/logs/{log or cmd.split()[2]}.log", "w") as fh:
        fh.write(f"# {cmd}\n# code {COMMIT}\n")
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end="")
            fh.write(line)
    if p.wait():
        raise RuntimeError(f"failed: {cmd}  (log: runs_3b/logs/)")

## 1. Retrieval (about 20 min)

Frozen CLIP embeddings, bi-encoder top-50, then the location re-ranker. Reference run (NewsVQA-lite, 500 test images, 1 seed): dev location R@1 **55.1** with re-ranker weight α = 0.1, test **56.3**. Within 1–2 points means retrieval is reproduced.

In [19]:
step("python -m nv.embed", done="emb/img_test_idx.json")
step("python -m nv.retrieve", done="results/retrieval_bi_recall.json")
step("python -m nv.cross_encoder", done="results/retrieval_ce_recall.json")
ce = json.load(open(f"{OUT}/results/retrieval_ce_recall.json"))
print("alpha:", ce.get("alpha"))
for s in ("dev", "test"):
    print(s, "location recall:", ce.get(s, {}).get("location"))

skip, already done: python -m nv.embed
skip, already done: python -m nv.retrieve
skip, already done: python -m nv.cross_encoder
alpha: 0.1
dev location recall: {'R@1': 55.05, 'R@5': 69.16, 'R@10': 75.73, 'R@50': 88.87}
test location recall: {'R@1': 56.14, 'R@5': 70.68, 'R@10': 75.61, 'R@50': 88.5}


## 2. Training and evaluation data (CPU, about 2 min)

6,000 training samples (QA / MCQA / QAR, 15% with wrong-place evidence) and evaluation prompts for every dev and test image that has a gold location or date. Reference run's training mix: 1,769 QA, 2,406 MCQA, 1,825 QAR, 909 noisy.

In [20]:
step("python -m nv.build_sft --n_train 6000 --n_eval 1644 --noise 0.15", done="sft/stats.json")
stats = json.load(open(f"{OUT}/sft/stats.json"))
N = {s: stats[s]["n_eval"] for s in ("dev", "test")}   # prompts per format, used to tell finished runs apart
print("train:", stats["train"]["formats"], "noisy:", stats["train"]["noisy"])
print("eval images:", N)

skip, already done: python -m nv.build_sft --n_train 6000 --n_eval 1644 --noise 0.15
train: {'qa': 1766, 'mcqa': 2410, 'qar': 1824} noisy: 907
eval images: {'dev': 1552, 'test': 1571}


## 3. Zero-shot baselines on test (about 45 min)

In [21]:
for fmt, name in [("qa_noev", "zs_noev"), ("qa", "zs_qa"), ("mcqa", "zs_mcqa")]:
    step(f"python -m nv.infer --split test --fmt {fmt} --name {name} --bs 16",
         done=f"preds/test/{name}.jsonl", n=N["test"], log=f"{name}_test")

skip, already done: python -m nv.infer --split test --fmt qa_noev --name zs_noev --bs 16
skip, already done: python -m nv.infer --split test --fmt qa --name zs_qa --bs 16
skip, already done: python -m nv.infer --split test --fmt mcqa --name zs_mcqa --bs 16


## 4. LoRA training (about 60–90 min)

LoRA r = 64 on Qwen2.5-VL-3B, 1 epoch, seed 0. The loss is printed every few steps and kept in `runs_3b/logs/train_lora_mixed.log` for the loss curve (`scripts/plot_loss.py`).

In [22]:
step("python -m nv.train_lora --out ckpt/lora_mixed", done="ckpt/lora_mixed/final/adapter_config.json",
     log="train_lora_mixed")

skip, already done: python -m nv.train_lora --out ckpt/lora_mixed


## 5. Fine-tuned model: test, plus dev for the trust layer (about 45 min)

The dev MCQA run gives the option probabilities that calibrate the trust layer's threshold.

In [23]:
for split, fmt, name in [("test", "qa", "lora_mixed_qa"), ("test", "mcqa", "lora_mixed_mcqa"),
                         ("dev", "mcqa", "lora_mixed_mcqa")]:
    step(f"python -m nv.infer --split {split} --fmt {fmt} --name {name} --bs 16 --adapter ckpt/lora_mixed/final",
         done=f"preds/{split}/{name}.jsonl", n=N[split], log=f"{name}_{split}")

skip, already done: python -m nv.infer --split test --fmt qa --name lora_mixed_qa --bs 16 --adapter ckpt/lora_mixed/final
skip, already done: python -m nv.infer --split test --fmt mcqa --name lora_mixed_mcqa --bs 16 --adapter ckpt/lora_mixed/final
skip, already done: python -m nv.infer --split dev --fmt mcqa --name lora_mixed_mcqa --bs 16 --adapter ckpt/lora_mixed/final


## 6. Trust layer (CPU, about 3 min)

Each answer backs off from city to region to country (day to month to year), or to no answer, until its confidence clears a threshold set so the expected error is at most eps (5%, 10%, 20%). Four rules are compared: `finest` always answers in full, `selective` answers in full or not at all, `backoff` uses the model's probabilities, and `evidence` also uses how many retrieved articles agree with the answer. One half of dev fits the evidence model, the other half sets the thresholds, and test scores. The second table repeats this on 200 random splits of dev + test to show how much one split can vary.

In [24]:
step("python -m nv.hierarchy --name lora_mixed_mcqa --resplits 200", log="trust_layer")

## Trust layer: `lora_mixed_mcqa`; dev halves fit the evidence model and set lambda, test scores

| dim | eps | method | lambda | error % | answered % | depth 1/2/3 % | specificity % |
|---|---|---|---|---|---|---|---|
| location | - | finest | 0.000 | 48.4 | 98.8 | 40/9/50 | 69.1 |
| location | 0.05 | selective | 0.677 | 6.0 | 29.4 | 16/2/10 | 17.6 |
| location | 0.05 | backoff | 0.751 | 5.8 | 40.2 | 29/4/7 | 19.6 |
| location | 0.05 | evidence | 0.809 | 6.3 | 60.4 | 51/7/2 | 23.8 |
| location | 0.1 | selective | 0.514 | 12.0 | 48.8 | 26/4/19 | 30.0 |
| location | 0.1 | backoff | 0.579 | 12.5 | 57.4 | 35/7/16 | 31.8 |
| location | 0.1 | evidence | 0.694 | 12.3 | 74.7 | 57/12/6 | 32.6 |
| location | 0.2 | selective | 0.333 | 20.4 | 63.4 | 31/5/28 | 41.3 |
| location | 0.2 | backoff | 0.372 | 21.1 | 72.9 | 40/7/26 | 44.0 |
| location | 0.2 | evidence | 0.521 | 20.3 | 86.0 | 57/14/15 | 43.4 |
| date | - | finest | 0.000 | 96.5 | 100.0 | 0/0/100 | 100.0 |
| date | 0.05 | selective | 0.884

## 7. Scores (CPU)

Reference run (NewsVQA-lite) on its 500-image subset: GREAT 51.1 (`zs_noev`), 57.2 (`zs_qa`), 62.5 (`lora_mixed_qa`). These numbers cover all test images with a gold answer, so expect small differences; the `GREAT@500` column scores the same 500 images as `results/`, and every GREAT comes with a 95% bootstrap interval and its gain over `zs_noev`.

In [25]:
step("python -m nv.evaluate --split test --baseline zs_noev", log="evaluate_test")
step(f"python scripts/extract_metrics.py --src {OUT}/results --out {OUT}/metrics_summary.json", log="extract_metrics")

8 runs, scored on 1571 shared images
## TARA test — 1571 images (location n=1476, date n=1303)

| run | date_EM@1 | date_E-F1 | Delta | GREAT_date | loc_EM@1 | loc_E-F1 | CO-delta | GREAT_loc | GREAT | MCQA_ECE |
|---|---|---|---|---|---|---|---|---|---|---|
| retrieval_bi | 3.8 | 43.4 | 41.7 | 58.9 | 21.8 | 53.0 | 23.1 | 46.6 | 52.8 |  |
| retrieval_ce | 3.9 | 43.2 | 42.1 | 59.5 | 24.7 | 57.6 | 26.4 | 51.6 | 55.6 |  |
| retrieval_ce_cluster | 3.4 | 43.6 | 41.0 | 58.6 | 24.7 | 57.6 | 26.4 | 51.6 | 55.1 |  |
| zs_noev | 2.4 | 43.1 | 41.1 | 59.8 | 27.6 | 51.2 | 29.8 | 45.0 | 52.4 |  |
| zs_qa | 3.9 | 47.1 | 43.8 | 56.5 | 30.9 | 61.6 | 33.5 | 57.9 | 57.2 |  |
| zs_mcqa | 3.5 | 41.6 | 40.7 | 54.8 | 24.5 | 57.5 | 26.8 | 52.0 | 53.4 | 24.8 |
| lora_mixed_mcqa | 4.6 | 48.6 | 50.8 | 64.5 | 45.1 | 65.2 | 46.6 | 61.0 | 62.8 | 19.2 |
| lora_mixed_qa | 4.3 | 47.7 | 49.0 | 64.0 | 45.5 | 65.4 | 47.0 | 61.3 | 62.7 |  |

### GREAT with 95% bootstrap interval (1,000 resamples of the 1571 images); `GREA

## 8. Before closing

Flush Drive so every file is saved, then **Runtime → Disconnect and delete runtime** to stop using compute units.

In [26]:
drive.flush_and_unmount()